# Testing Efficiency and Effectiveness of a Deep Learning CNN Model  
## Fundoscopic optic nerve head images: diabetes vs non-diabetes

This notebook evaluates a trained convolutional neural network (CNN) classifier using both **effectiveness metrics** and **efficiency metrics**.

**Effectiveness** asks: *How well does the model classify diabetic vs non-diabetic optic nerve head images?*  
**Efficiency** asks: *How expensive is the model to run in time, memory, and computational cost?*

Assumed dataset layout:

```text
data/
├── train/
│   ├── diabetes/
│   └── no_diabetes/
├── val/
│   ├── diabetes/
│   └── no_diabetes/
└── test/
    ├── diabetes/
    └── no_diabetes/
```

This notebook can be used either after training a CNN or to test a saved CNN model.

## Library list and what each library does

| Library | Purpose in this notebook |
|---|---|
| `pathlib` | Handles file and folder paths cleanly. |
| `time` | Measures inference latency and throughput. |
| `numpy` | Numerical arrays and vectorized calculations. |
| `pandas` | Creates tables of predictions and metric summaries. |
| `matplotlib` | Plots confusion matrix, ROC curve, PR curve, calibration curve, and training curves. |
| `torch` | Runs the CNN model and tensor operations. |
| `torchvision` | Loads images, applies transforms, and optionally provides pretrained CNN architectures. |
| `sklearn.metrics` | Computes standard validation/test metrics such as accuracy, precision, recall, F1, ROC-AUC, PR-AUC, log loss, and confusion matrix. |
| `sklearn.calibration` | Generates calibration curves to test whether predicted probabilities are trustworthy. |

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    log_loss,
    brier_score_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)
from sklearn.calibration import calibration_curve

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## Step 1 — Define paths and basic settings

This step defines where the image data are stored and sets the image size and batch size used for testing.

For optic nerve head fundus images, resizing is necessary so that all images have the same tensor shape. Use the same image size used during training.

In [ ]:
DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

CLASS_NAMES = ["diabetes", "no_diabetes"]  # adjust if your folders use different names
POSITIVE_CLASS = "diabetes"

## Step 2 — Define image transforms

The test/validation transform should be deterministic: resize, center crop if needed, convert to tensor, and normalize.

Do **not** use random augmentation on validation or test data, because the goal is to measure true model performance on fixed images.

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## Step 3 — Load train, validation, and test datasets

This step creates PyTorch datasets and dataloaders.

The **train loader** is used for training or checking training-set behavior.  
The **validation loader** is used for model selection and tuning.  
The **test loader** should be used only once at the end for final unbiased evaluation.

In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds = datasets.ImageFolder(VAL_DIR, transform=test_transform)
test_ds = datasets.ImageFolder(TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Class mapping:", train_ds.class_to_idx)
print("Train images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Test images:", len(test_ds))

## Step 4 — Load or define the CNN model

This example uses a ResNet-18 CNN with a two-class output layer.

For medical imaging, transfer learning is usually a strong baseline because pretrained CNNs already know useful low-level visual features such as edges, textures, vessels, and contrast patterns.

Replace `MODEL_PATH` with the path to your trained model weights.

In [ ]:
NUM_CLASSES = 2
MODEL_PATH = Path("trained_fundus_cnn.pt")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

if MODEL_PATH.exists():
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Loaded model weights from {MODEL_PATH}")
else:
    print("No saved model found. This model is randomly initialized until you load trained weights.")

model = model.to(DEVICE)
model.eval()

## Step 5 — Generate predictions on a dataloader

This function runs the CNN on all images and returns:

- true labels
- predicted labels
- predicted probability for the positive class
- raw class probabilities
- image paths

The positive-class probability is needed for ROC-AUC, PR-AUC, log loss, calibration, and threshold analysis.

In [ ]:
@torch.no_grad()
def predict_loader(model, loader, positive_class_name=POSITIVE_CLASS):
    model.eval()
    y_true = []
    y_pred = []
    y_prob_pos = []
    y_prob_all = []
    image_paths = []

    class_to_idx = loader.dataset.class_to_idx
    pos_idx = class_to_idx[positive_class_name]

    start_idx = 0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = labels.size(0)
        paths = [loader.dataset.samples[i][0] for i in range(start_idx, start_idx + batch_size)]
        start_idx += batch_size

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob_pos.extend(probs[:, pos_idx].cpu().numpy())
        y_prob_all.extend(probs.cpu().numpy())
        image_paths.extend(paths)

    return {
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "y_prob_pos": np.array(y_prob_pos),
        "y_prob_all": np.array(y_prob_all),
        "image_paths": image_paths,
        "class_to_idx": class_to_idx,
    }

val_pred = predict_loader(model, val_loader)
test_pred = predict_loader(model, test_loader)

## Step 6 — Compute standard effectiveness metrics

These are the most commonly reported classification metrics.

| Metric | What it tells you |
|---|---|
| Accuracy | Fraction of all images classified correctly. Can be misleading if classes are imbalanced. |
| Balanced accuracy | Average of sensitivity and specificity; better when class sizes differ. |
| Precision | Among images predicted diabetic, how many were truly diabetic. High precision means fewer false positives. |
| Recall / sensitivity | Among truly diabetic images, how many were detected. High recall means fewer false negatives. |
| Specificity | Among truly non-diabetic images, how many were correctly rejected. |
| F1 score | Harmonic mean of precision and recall. Useful when false positives and false negatives both matter. |
| ROC-AUC | Measures ranking ability across all thresholds. Higher means diabetic images tend to get higher scores than non-diabetic images. |
| PR-AUC | Precision-recall area. Especially useful when the positive class is rare. |
| Log loss | Penalizes confident wrong predictions. Lower is better. |
| Brier score | Measures probability calibration error. Lower is better. |
| MCC | Matthews correlation coefficient; robust single-number metric for imbalanced binary classification. |
| Cohen kappa | Agreement beyond chance. |

In [ ]:
def compute_binary_metrics(pred_dict):
    y_true = pred_dict["y_true"]
    y_pred = pred_dict["y_pred"]
    y_prob = pred_dict["y_prob_pos"]
    class_to_idx = pred_dict["class_to_idx"]
    pos_idx = class_to_idx[POSITIVE_CLASS]

    y_true_bin = (y_true == pos_idx).astype(int)
    y_pred_bin = (y_pred == pos_idx).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    metrics = {
        "accuracy": accuracy_score(y_true_bin, y_pred_bin),
        "balanced_accuracy": balanced_accuracy_score(y_true_bin, y_pred_bin),
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall_sensitivity": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
        "roc_auc": roc_auc_score(y_true_bin, y_prob),
        "pr_auc_average_precision": average_precision_score(y_true_bin, y_prob),
        "log_loss": log_loss(y_true_bin, y_prob),
        "brier_score": brier_score_loss(y_true_bin, y_prob),
        "mcc": matthews_corrcoef(y_true_bin, y_pred_bin),
        "cohen_kappa": cohen_kappa_score(y_true_bin, y_pred_bin),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
    }
    return pd.DataFrame(metrics, index=[0]).T.rename(columns={0: "value"})

val_metrics = compute_binary_metrics(val_pred)
test_metrics = compute_binary_metrics(test_pred)

display(val_metrics.rename(columns={"value": "validation"}))
display(test_metrics.rename(columns={"value": "test"}))

## Step 7 — Confusion matrix

The confusion matrix shows the actual error types:

- **True positive**: diabetic image correctly classified as diabetic
- **False positive**: non-diabetic image incorrectly classified as diabetic
- **True negative**: non-diabetic image correctly classified as non-diabetic
- **False negative**: diabetic image incorrectly classified as non-diabetic

For medical screening, false negatives may be especially important because they represent missed disease-associated cases.

In [ ]:
def plot_confusion_matrix(pred_dict, title="Confusion Matrix"):
    y_true = pred_dict["y_true"]
    y_pred = pred_dict["y_pred"]
    class_to_idx = pred_dict["class_to_idx"]
    pos_idx = class_to_idx[POSITIVE_CLASS]
    y_true_bin = (y_true == pos_idx).astype(int)
    y_pred_bin = (y_pred == pos_idx).astype(int)
    cm = confusion_matrix(y_true_bin, y_pred_bin)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No diabetes", "Diabetes"])
    ax.set_yticklabels(["No diabetes", "Diabetes"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(test_pred, "Test Confusion Matrix")

## Step 8 — ROC curve

The ROC curve shows the tradeoff between sensitivity and false-positive rate across all possible probability thresholds.

- A model with ROC-AUC = 0.5 is no better than random ranking.
- A model with ROC-AUC = 1.0 perfectly ranks all positives above all negatives.

ROC-AUC is useful, but it can look overly optimistic when the positive class is rare.

In [ ]:
def plot_roc(pred_dict, title="ROC Curve"):
    y_true = pred_dict["y_true"]
    y_prob = pred_dict["y_prob_pos"]
    pos_idx = pred_dict["class_to_idx"][POSITIVE_CLASS]
    y_true_bin = (y_true == pos_idx).astype(int)

    fpr, tpr, thresholds = roc_curve(y_true_bin, y_prob)
    auc_value = roc_auc_score(y_true_bin, y_prob)

    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate / sensitivity")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_roc(test_pred, "Test ROC Curve")

## Step 9 — Precision-recall curve

The precision-recall curve is often more informative than ROC when classes are imbalanced.

- Precision answers: when the model says diabetic, how often is it correct?
- Recall answers: how many diabetic cases does it catch?

PR-AUC is especially useful when the diabetic group is smaller than the non-diabetic group.

In [ ]:
def plot_pr(pred_dict, title="Precision-Recall Curve"):
    y_true = pred_dict["y_true"]
    y_prob = pred_dict["y_prob_pos"]
    pos_idx = pred_dict["class_to_idx"][POSITIVE_CLASS]
    y_true_bin = (y_true == pos_idx).astype(int)

    precision, recall, thresholds = precision_recall_curve(y_true_bin, y_prob)
    ap = average_precision_score(y_true_bin, y_prob)

    plt.figure(figsize=(5, 4))
    plt.plot(recall, precision, label=f"PR-AUC/AP = {ap:.3f}")
    plt.xlabel("Recall / sensitivity")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_pr(test_pred, "Test Precision-Recall Curve")

## Step 10 — Calibration curve

Calibration asks whether predicted probabilities are meaningful.

Example: among images where the model predicts 0.80 probability of diabetes, approximately 80% should actually be diabetic.

A model can have high ROC-AUC but poor calibration. Calibration matters if model probabilities will be used for confidence, triage, or decision thresholds.

In [ ]:
def plot_calibration(pred_dict, title="Calibration Curve"):
    y_true = pred_dict["y_true"]
    y_prob = pred_dict["y_prob_pos"]
    pos_idx = pred_dict["class_to_idx"][POSITIVE_CLASS]
    y_true_bin = (y_true == pos_idx).astype(int)

    frac_pos, mean_pred = calibration_curve(y_true_bin, y_prob, n_bins=10, strategy="uniform")

    plt.figure(figsize=(5, 4))
    plt.plot(mean_pred, frac_pos, marker="o", label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed fraction positive")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_calibration(test_pred, "Test Calibration Curve")

## Step 11 — Threshold analysis

The default threshold is usually 0.5, but medical applications often require a custom threshold.

For example:

- Lower threshold: higher sensitivity, fewer missed diabetic cases, more false positives.
- Higher threshold: higher specificity, fewer false positives, more missed diabetic cases.

This step evaluates performance over many thresholds so you can choose a threshold based on the intended use.

In [ ]:
def threshold_table(pred_dict, thresholds=np.linspace(0.05, 0.95, 19)):
    y_true = pred_dict["y_true"]
    y_prob = pred_dict["y_prob_pos"]
    pos_idx = pred_dict["class_to_idx"][POSITIVE_CLASS]
    y_true_bin = (y_true == pos_idx).astype(int)

    rows = []
    for t in thresholds:
        y_pred_bin = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        rows.append({
            "threshold": t,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        })
    return pd.DataFrame(rows)

thresh_df = threshold_table(test_pred)
display(thresh_df)

plt.figure(figsize=(6, 4))
plt.plot(thresh_df["threshold"], thresh_df["sensitivity"], marker="o", label="Sensitivity")
plt.plot(thresh_df["threshold"], thresh_df["specificity"], marker="o", label="Specificity")
plt.plot(thresh_df["threshold"], thresh_df["f1"], marker="o", label="F1")
plt.xlabel("Probability threshold")
plt.ylabel("Metric value")
plt.title("Threshold tradeoff")
plt.legend()
plt.tight_layout()
plt.show()

## Step 12 — Save per-image predictions

This step writes one row per image so you can review individual errors.

This is useful for checking whether mistakes are associated with poor focus, low illumination, off-center optic nerve head, media opacity, artifacts, or mislabeled data.

In [ ]:
def predictions_dataframe(pred_dict):
    idx_to_class = {v: k for k, v in pred_dict["class_to_idx"].items()}
    df = pd.DataFrame({
        "image_path": pred_dict["image_paths"],
        "true_label_idx": pred_dict["y_true"],
        "pred_label_idx": pred_dict["y_pred"],
        "prob_diabetes": pred_dict["y_prob_pos"],
    })
    df["true_label"] = df["true_label_idx"].map(idx_to_class)
    df["pred_label"] = df["pred_label_idx"].map(idx_to_class)
    df["correct"] = df["true_label_idx"] == df["pred_label_idx"]
    return df

test_predictions_df = predictions_dataframe(test_pred)
test_predictions_df.to_csv("test_predictions.csv", index=False)
display(test_predictions_df.head())

## Step 13 — Training/validation curve interpretation

If you saved a training history CSV, load it here.

Common signs:

| Pattern | Interpretation |
|---|---|
| Train loss decreases, validation loss decreases | Healthy learning. |
| Train loss decreases, validation loss increases | Overfitting. |
| Both losses remain high | Underfitting or learning-rate/problem issue. |
| Validation metric unstable | Dataset may be small, noisy, imbalanced, or augmented too aggressively. |

In [ ]:
HISTORY_PATH = Path("training_history.csv")

if HISTORY_PATH.exists():
    history = pd.read_csv(HISTORY_PATH)
    display(history.head())

    if {"epoch", "train_loss", "val_loss"}.issubset(history.columns):
        plt.figure(figsize=(6, 4))
        plt.plot(history["epoch"], history["train_loss"], label="Train loss")
        plt.plot(history["epoch"], history["val_loss"], label="Validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training and validation loss")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No training_history.csv found. Skip this section or save training curves during model training.")

## Step 14 — Efficiency metrics

Efficiency describes whether the CNN is practical to deploy.

| Metric | What it tells you |
|---|---|
| Parameter count | Number of trainable weights; larger models require more storage and may overfit small datasets. |
| Model size | Disk space required for the saved model. |
| Latency | Time to classify one image. Important for real-time or clinical workflow use. |
| Throughput | Images processed per second. Important for batch screening. |
| GPU memory use | Whether the model fits comfortably on available hardware. |

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

if MODEL_PATH.exists():
    size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)
    print(f"Saved model size: {size_mb:.2f} MB")

In [ ]:
@torch.no_grad()
def benchmark_inference(model, loader, warmup_batches=3, max_batches=20):
    model.eval()

    # Warmup
    for i, (images, _) in enumerate(loader):
        if i >= warmup_batches:
            break
        images = images.to(DEVICE)
        _ = model(images)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    n_images = 0
    start = time.perf_counter()

    for i, (images, _) in enumerate(loader):
        if i >= max_batches:
            break
        images = images.to(DEVICE)
        _ = model(images)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        n_images += images.size(0)

    elapsed = time.perf_counter() - start
    latency_ms = 1000 * elapsed / n_images
    throughput = n_images / elapsed

    return {
        "images_tested": n_images,
        "elapsed_seconds": elapsed,
        "latency_ms_per_image": latency_ms,
        "throughput_images_per_second": throughput,
    }

benchmark = benchmark_inference(model, test_loader)
pd.DataFrame(benchmark, index=[0]).T.rename(columns={0: "value"})

## Step 15 — Error analysis

This step separates false positives and false negatives.

For optic nerve head images, manually reviewing errors is important because misclassifications may reflect:

- poor focus
- shadows or glare
- off-center optic disc
- blood vessel artifacts
- variable field of view
- labeling problems
- disease confounders not related to diabetes

In [ ]:
errors_df = test_predictions_df[~test_predictions_df["correct"]].copy()
false_negatives = errors_df[(errors_df["true_label"] == POSITIVE_CLASS) & (errors_df["pred_label"] != POSITIVE_CLASS)]
false_positives = errors_df[(errors_df["true_label"] != POSITIVE_CLASS) & (errors_df["pred_label"] == POSITIVE_CLASS)]

print("Total errors:", len(errors_df))
print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))

display(false_negatives.head())
display(false_positives.head())

## Step 16 — Interpretability note: Grad-CAM

For a CNN trained on fundus images, it is not enough to know that the model is accurate. You should also check whether it is looking at biologically plausible image regions.

Grad-CAM can show which image regions contributed most to the prediction.

For this dataset, useful checks include:

- Is attention centered on the optic nerve head?
- Is the model using retinal vessels, illumination artifacts, borders, or labels instead of anatomy?
- Are diabetic predictions driven by plausible optic nerve head or peripapillary features?

Recommended library: `pytorch-grad-cam`.

Install if needed:

```bash
pip install grad-cam
```

This section is left as a recommended add-on because Grad-CAM implementation depends on the exact CNN architecture and layer names.

## Step 17 — Final metric summary export

This step saves the validation and test metric tables to CSV files.

Use validation metrics for model selection. Use test metrics for the final reported performance.

In [ ]:
val_metrics.to_csv("validation_metrics.csv")
test_metrics.to_csv("test_metrics.csv")
thresh_df.to_csv("threshold_analysis.csv", index=False)

print("Saved:")
print("- validation_metrics.csv")
print("- test_metrics.csv")
print("- threshold_analysis.csv")
print("- test_predictions.csv")

# Summary: what to report

For a CNN classifier of diabetes vs non-diabetes from optic nerve head fundus images, report:

1. Dataset sizes for train, validation, and test sets  
2. Class balance  
3. Accuracy and balanced accuracy  
4. Sensitivity and specificity  
5. Precision and F1 score  
6. ROC-AUC and PR-AUC  
7. Confusion matrix  
8. Calibration metrics: Brier score and calibration curve  
9. Threshold analysis  
10. Efficiency metrics: model size, parameters, latency, throughput  
11. Error analysis and image-quality review  
12. Optional Grad-CAM interpretability review  

For medical-style image classification, **sensitivity, specificity, ROC-AUC, PR-AUC, calibration, and error analysis** are usually more informative than accuracy alone.